# Metrics: diversity

In this module, we take a closer look at **diversity** and **filter bubbles** as evaluation concepts for recommender systems. This notebook also serves as a **model for the final project**, so pay close attention to the overall setup and reasoning.

We do **not** structure this notebook as a fully publishable academic paper. We do not perform extensive statistical analyses or strictly follow established methodologies (and we do not expect that from you in the final project either). Instead, we approach this as a **pilot study**: the goal is to explore whether a particular experimental setup shows *any promise at all* before investing effort into more elaborate analyses.

We begin with the central research question:

**Can we find evidence of a filter bubble effect in a real-life news recommender system?**

To investigate this, we will use recommendation and interaction data from the **MIND dataset**, which is based on real-world news consumption behavior.

There is a lot of research investigating data from different recommender systems, trying to determine whether there is evidence for **filter bubble effects**. One example is:

Nguyen, Tien T., et al. [*"Exploring the filter bubble: the effect of using recommender systems on content diversity."*](https://scholar.google.nl/scholar?hl=en&as_sdt=0%2C5&q=Exploring+the+Filter+Bubble%3A+The+Effect+of+Using+Recommender+Systems+on+Content+Diversity&btnG=#:~:text=UvA%2Dlinker%3A%20full%20text) Proceedings of the 23rd international conference on World wide web. 2014.

> Read this paper!

Because useful data on news consumption was not available at the time, the authors used **MovieLens data** as a proxy. This relies on a very strong assumption: that patterns in movie-watching behavior are sufficiently similar to patterns in news consumption to allow insights about diversity effects.

> Think about this: Is this a reasonable assumption?

The paper relies on a **genome-based diversity metric**. In this context, the *genome* of an item (movie or article) is a vector representation of its content. Each dimension corresponds to a keyword, and the value along that dimension indicates how strongly that attribute is associated with the item. Once items are represented as genome vectors, diversity can be measured by examining how **similar or dissimilar** a set of recommended items is. For example:

* If a recommender repeatedly suggests items with very similar genome vectors, diversity is low.
* If suggested items span a wide range of topics (i.e., their vectors are more dissimilar), diversity is higher.

## Detecting filter bubble effects

The authors attempt to detect filter bubble effects by looking at **user preferences over time**. The idea is that if diversity decreases over time, users may have become trapped in a filter bubble. For instance, if users start out being interested in diverse content, but after months of using the platform the diversity of recommended content decreases, this could indicate that the algorithm has pushed them into a niche.

> Think about this: Is this a valid interpretation of a filter bubble?

## Disclaimer

Beware that we do **not** use this paper as an example of best practice for structuring this type of research. Feel free to reflect critically on it and form your own opinion :).

The main reason we use this paper that it is quite straightforward to reproduce some of its methods. This makes it useful as a starting point for exploring diversity metrics and filter bubble effects, even if the conclusions and assumptions of the paper itself should be taken with a grain of salt.


## Reproduce

Ideally, we would like to reproduce the ideas from this paper using **actual news data** (e.g., MIND). However, it turns out to be nearly impossible to find data that tracks enough individual users over a sufficiently long period to reliably observe diminishing diversity effects.

> This might seem like only a nuisance for us, but as this article suggests, the lack of public data may point to a more fundamental problem:
>
> van Drunen, Max, and Sanne Vrijenhoek. [*“How Public Datasets Constrain the Development of Diversity-Aware News Recommender Systems, and What Law Could Do About It.”*](https://scholar.google.nl/scholar?hl=en&as_sdt=0%2C5&q=How+public+datasets+constrain+the+development+of+diversity-aware+news+recommender+systems%2C+and+what+law+could+do+about+it.&btnG=#:~:text=include%20citations-,%5BPDF%5D%20arxiv.org,-How%20public%20datasets) arXiv preprint arXiv:2510.05952 (2025).
>
> They argue that as a result, important questions about diversity, fairness, etc. often remain empirically underexplored (not because they are unimportant, but because the data needed to study them is not available to the public).

So we cannot completely reproduce the study above, but what we *can* do is examine whether diversity-related effects are visible in the data at all. For example:

**Are the articles recommended by the algorithm less diverse than articles selected at random?**

To answer this question, we need to reproduce the diversity measure used in the paper.

## Genome

To do this, we first need to compute the genome of all articles. Unfortunately, the paper by Nguyen et al. does not specify exactly how the genome is computed, so we have to make some assumptions.

However, we have essentially done this before. It is reasonable to interpret the genome as the output of an NLP technique such as **TF–IDF** or **word-embedding–based representations** (such as Word2Vec).

The exact method matters less than the core idea: representing items in a semantic space that allows us to quantify similarity and, by extension, **diversity**.

To simplify matters, we have pre-computed the genome using simple word embeddings (similar to the Word2Vec approach from the previous module). You already know how to do this, but computing the genome for the full dataset takes a very long time (around three days on my computer), so this step was done in advance.

## Diversity

In this study, content diversity for a set of items (articles in our case, movies in theirs) is computed by calculating the **pairwise Euclidean distance** between all items in the set and then taking the **average** of those distances. We will work this out in more detail below.





# Getting started

Start by loading the necessary libraries. And loading the data.

In [ ]:
import pandas as pd
import numpy as np
import pooch
from tqdm.notebook import tqdm

%reload_ext autoreload
%autoreload 2

# Download data
DATA_REPO = "https://raw.githubusercontent.com/uvapl/recommender-systems/2025/data/m4/"
for fname in ["MIND_news.tsv", "MIND_behaviors.tsv", "MIND_genome.csv"]:
    pooch.retrieve(url = DATA_REPO + fname, known_hash=None, fname=fname, path="data", progressbar=True)

# Read data
# Column names (see MIND documentation)
columns_articles = ["NewsID", "Category", "SubCategory", "Title", "Abstract", "URL", "TitleEntities", "AbstractEntities"]
news_df = pd.read_csv("data/MIND_news.tsv", sep="\t", index_col = "NewsID", header=None, names=columns_articles, encoding='utf-8')[["Title", "Abstract"]]

# Column names (see MIND documentation)
columns_impressions = ["ImpressionID", "UserID", "Time", "History", "Impressions"]
impressions = pd.read_csv("data/MIND_behaviors.tsv", sep="\t", index_col = "ImpressionID", header=None, names=columns_impressions, encoding='utf-8')

genome = pd.read_csv("data/MIND_genome.csv", index_col="NewsID", na_values = ["-999.0000"]).dropna()

Inspect the data for yourself, here below. Pay special attention to what the *genome* DataFrame looks like.

# Euclidean distance

The study relies on the **Euclidean distance** between genome vectors. This is a *distance metric* (more similar vectors -> lower values), not a *similarity measure* (more similar vectors -> higher values, such as cosine similarity).

You already know Euclidean distance in the 2D case from the Pythagorean theorem. For example, consider two items (articles) described by two genome features:

| Item | feature1 | feature2 |
| ---- | -------- | -------- |
| N001 | 0.1      | 0.2      |
| N002 | -0.1     | 0.3      |

The Euclidean distance between them is:

$$
d(N001, N002) = \sqrt{(0.1 - (-0.1))^2 + (0.2 - 0.3)^2}
= \sqrt{(0.2)^2 + (-0.1)^2}
= \sqrt{0.05}
\approx 0.224
$$

For vectors with any number of features
$a = (a_1, a_2, a_3, \ldots)$ and
$b = (b_1, b_2, b_3, \ldots)$,
this generalizes to:

$$
d(a, b) = \sqrt{(a_1 - b_1)^2 + (a_2 - b_2)^2 + (a_3 - b_3)^2 + \ldots}
$$

If you remember some linear algebra, you may recognize this in vector notation as:

$$
d(a, b) = \sqrt{(a - b) \cdot (a - b)}
$$

Which is simply the **norm of the difference vector**:

$$
d(a, b) = \lVert a - b \rVert
$$


### Question 1

*3 pts*

Complete the `euclidean_distance()` function below. It takes two Pandas Series (representing genome vectors for two items) as input and returns the **Euclidean distance** between them as a single float value.

As good practice for future assignments, rely on the **vector definition** of the Euclidean distance:

$$
d(a, b) = \sqrt{(a - b) \cdot (a - b)}
$$

So, do **not** use a `for` loop in your solution. Instead, use vector operations and the Pandas `@` operator.


In [ ]:
def euclidean_distance(a: pd.Series, b: pd.Series) -> float:
    # your code here

d = euclidean_distance(genome.loc["N55528"], genome.loc["N38324"])
print(f"Euclidean distance between articles N55528 and N38324: {d:.3f}")

### Question 2

*6 pts*

Complete the `compute_mean_distance()` function below. The goal of this function is to compute the **mean Euclidean distance** between a selected set of articles (typically between 5 and 20).

The input is a Pandas DataFrame containing a subset of rows from the `genome` DataFrame. The function should compute the **pairwise Euclidean distance** between all distinct pairs of rows and return the **mean** of those distances.

Make sure to **exclude** distances between a row and itself (which are always 0), as they do not count towards the mean.

In [ ]:
def compute_mean_distance(genomes: pd.DataFrame):
    # your code here

sample = genome.iloc[:10]
mean = compute_mean_distance(sample)
print(f"Mean Euclidean distance of first 10 articles in genome: {mean:.3f}")

# Baseline

Before looking at the diversity of the recommendations, we first create a **baseline**. In this baseline, we simulate **500 sessions**, where in each session we “recommend” **20 completely random articles**.

Just as in the study above, we use the **mean pairwise Euclidean distance** as our diversity measure. Concretely, we compute the mean distance within each session and then take the mean of those values across all simulated sessions.

This gives us a reference point for the **overall diversity of the article pool**, without any influence from a recommender system. Run the code below (this might take a few minutes):

In [ ]:
N = 500
M = 20

total = 0
for i in tqdm(range(N)):
    sample = genome.sample(M)
    total += compute_mean_distance(sample)

print(f"Baseline: mean Euclidean distance (diversity) of {N} sessions with {M} randomly recommended articles: {total/N:.3f}")

The resulting baseline (diversity) above should be around 1.19.

# Recommender System Results

The `impressions` DataFrame contains all the information we need from the actual (Microsoft News) recommender system.

The `Impressions` column lists all articles shown in each session. The suffix `-1` or `-0` indicates whether the user **clicked** the article (`-1`) or **ignored** it (`-0`). However, this format is not very convenient for our analysis, so we first need to transform it:

* We do **not** need session-level information. Instead, for each user we want to know:
  * which articles were **recommended** (across all sessions),
  * which articles were **clicked** (across all sessions),
  * which articles were **ignored** (across all sessions).
* We want the article IDs as actual **Python lists** (not one long string).
* We want to remove the `-1` / `-0` suffixes and store clicked and ignored articles in **separate columns**.

### Question 3

*8 pts.*

Complete the function `transform()` below. It takes the `impressions` DataFrame, and a set of article IDs for the **known articles** (i.e., articles for which we have genome data).

It should return a DataFrame with:

* `UserID` as the index, and
* three columns: **`Impressions`**, **`Clicked`**, and **`Ignored`**.

Each column should contain a list of article IDs:

* `Impressions`: all recommended articles (clicked **and** ignored),
* `Clicked`: only clicked (`-1`),
* `Ignored`: only ignored (`-0`).

The lists should only contain those articles that are in the `know_articles` set (as those are the only articles we can compare). 

(!) Beware that most users have **multiple sessions**, so you will need to **combine** impressions across sessions for the same user. 


In [ ]:
def transform(df: pd.DataFrame, known_articles: set) -> (pd.DataFrame, pd.Series):
    # your code here

known_articles = set(genome.index)
impressions_expanded = transform(impressions, known_articles)
print(impressions_expanded.head())

# Select active users

In order to get a sense of the diversity of recommended and clicked items we need to use data of users that were relatively active (if they only read one or two articles we clearly don't have enough inormation). So let's only select users that read at least 5 articles.

In [ ]:
impressions_selected = impressions_expanded[impressions_expanded["Clicked"].apply(len) > 5]
impressions_selected

### Question 4

*6 pts.*

Finish the function `compute_mean_distance_data()` below.

As input, it takes a selected sample of the transformed impressions data (we will use **100 random users**—running this on the full dataset would take hours). For each user in this sample, the function should:

* Select the list of article IDs from the specified `column` (either `"Clicked"` or `"Impressions"`).
* Use `compute_mean_distance()` (from the previous question) to compute the mean pairwise Euclidean distance between the genome vectors of those articles.
* If the list contains more than 20 articles, use a **random sample of 20** articles instead, to keep runtime manageable.

Finally, the function should return a single diversity score, defined as the average of the per-user mean distances.

*Why is it reasonable to take the “mean of means” here?*

*Computational note: `compute_mean_distance()` has time complexity $O(n^2)$ because it computes distances for all pairs of items, so runtime grows quickly as the number of articles to compare becomes too large.

In [ ]:
def compute_mean_distance_data(selected_data, column = "Clicked"):
    # your code here

diversity_clicked = compute_mean_distance_data(impressions_selected.sample(100), column = "Clicked")
diversity_recommended = compute_mean_distance_data(impressions_selected.sample(100), column = "Impressions") 

print(f'Diversity of clicked "liked" articles: {diversity_clicked:.3f}')
print(f'Diversity of clicked "liked" articles: {diversity_clicked:.3f}')

You ran this analysis on only a sample of 100 items to keep the computation time reasonable. We ran the same procedure on the full dataset and obtained the following results:

* **Clicked items:** diversity ≈ **1.154**
* **All recommended items:** diversity ≈ **1.204**
* **Baseline (random articles):** diversity ≈ **1.19**

These results indicate that the articles **recommended by Microsoft’s recommender system are not less diverse than the baseline**, which represents the overall diversity of articles in the dataset. We have not performed formal significance tests, but given that this is a *negative result* (i.e., no clear decrease in diversity), such tests are not really helpful, right now.

We do see a *slightly* lower diversity among clicked items. A natural explanation is that, even when users are presented with a diverse set of articles, their **personal preferences** may lead them to click on a somewhat narrower subset. In that case, the reduced diversity would reflect *user choice* rather than the effect of the recommender system itself.

Of course, drawing strong conclusions would require a much more in-depth analysis with data over a longer time period and proper statistical testing.


# Conclusion

So, does the data suggest that it would be useful to run the full experiment described in the paper from the start? Think about this yourself.

I can think of one good reason why not: we do **not** observe noticeably lower diversity in the **recommended** articles compared to the random baseline. (And we see only a slightly lower diversity in the **clicked** articles, but that difference is more easily explained by **user selectivity** than by the recommender system itself.)

That is, the fact that we do *not* see reduced diversity in the recommended set suggests that the recommender system may not be strongly affecting diversity here, which makes it less likely that we would find a pronounced “diminishing diversity over time” effect, even if we had the data to run that experiment.

Why might that be? Try to think of some reasons why this recommender system might not affect diversity that much.

Now before you draw any strong conclusions about this data or recommender systems in general. There are many assumptions and limitations that make strong conclusions difficult. To name only a few obvious ones:

* **Genome construction.** We created our own genome representation. The original paper does not fully specify how their genome was computed, and our representation may not capture article semantics in the same way. (That said, the fact that we observe *some* decrease in diversity for clicked articles suggests that the genome has at least some validity.)
* **User sample bias.** We restricted the analysis to the most active users because we do not have enough interaction data for less active users. These users may not be representative of the broader population.
* **Clicking isn't "liking".** Clicks are not a perfect proxy for “liking”.

Given these constraints, we cannot draw any strong conclusions. But as a small **pilot investigation**, the results do not show particularly strong promise for further experimentation. Unless there were other compelling theoretical or empirical reasons to believe that this recommender system meaningfully affects diversity, I would not, as a researcher, be very inclined to continue this specific line of investigation.

## Take-home message

An important methodological lesson: if you are planning any experiment or analysis, **do the easy things first**. Run sanity checks and baseline comparisons before investing time in elaborate and computationally expensive experiments. You might not need to run them at all. (Keep this in mind for the final project.)
